# Chunk Regeneration Classifier

Обучаем chunk-level классификаторы, которые по выбранным model-only chunk metrics предсказывают, стоит ли перегенерировать сгенерированный action chunk.

Target: `needs_regeneration = 1` для chunks из failure-эпизодов, `0` для chunks из success-эпизодов.

Важно: split делается по `total_episode_idx`, чтобы chunks одного эпизода не попадали одновременно в train и validation.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    display
except NameError:
    def display(obj):
        print(obj.to_string(index=False) if hasattr(obj, 'to_string') else obj)

RANDOM_STATE = 42
METRICS_DIR = Path('/home/motovilovil/Robotics/robotics_project/mimic-video/eval_outputs/libero_spatial_one_10tasks_10eps_device3_trace_modelonly/metrics')
OUTPUT_DIR = METRICS_DIR / 'chunk_regeneration_classifiers'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = [
    'chunk_action_delta_norm_mean', 'flow_prediction_error',
    'action_chunk_temporal_consistency', 'plan_drift',
    'action_entropy', 'video_action_mutual_information',
    'mlp_activation_mean', 'task_semantic_alignment',
    'video_latent_variance_mean', 'representation_drift_rate',
    'decoder_hidden_norm_mean', 'object_interaction_confidence',
    'video_latent_norm_std', 'sampling_stability',
    'chunk_action_variance', 'video_latent_entropy',
    'decoder_hidden_norm_std', 'mlp_sparsity',
    'video_action_alignment_score', 'video_action_cosine_similarity',
    'goal_latent_distance', 'latent_success_alignment',
    'residual_stream_norm', 'chunk_action_delta_norm_max',
    'attention_sparsity', 'video_latent_delta_l2_std',
    'action_chunk_smoothness', 'action_uncertainty',
    'instruction_attention_score', 'chunk_gripper_switches',
    'latent_oscillation_score', 'video_latent_cosine_initial_final',
    'video_latent_norm_mean', 'score_norm_mean',
    'latent_path_efficiency', 'query_latency_sec',
]

def load_data(metrics_dir: Path) -> pd.DataFrame:
    csv_path = metrics_dir / 'chunk_metrics.csv'
    jsonl_path = metrics_dir / 'chunk_metrics.jsonl'
    if csv_path.exists():
        df = pd.read_csv(csv_path)
    elif jsonl_path.exists():
        df = pd.read_json(jsonl_path, lines=True)
    else:
        raise FileNotFoundError(f'No chunk metrics found in {metrics_dir}')
    df['success'] = df['success'].map(lambda v: v is True or str(v).strip().lower() in {'true', '1', 'yes'})
    df['needs_regeneration'] = (~df['success']).astype(int)
    missing = [col for col in FEATURES if col not in df.columns]
    if missing:
        raise ValueError(f'Missing feature columns: {missing}')
    for col in FEATURES:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

df = load_data(METRICS_DIR)
episode_labels = df.groupby('total_episode_idx')['needs_regeneration'].max().reset_index()
train_episodes, val_episodes = train_test_split(
    episode_labels['total_episode_idx'],
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=episode_labels['needs_regeneration'],
)
train_df = df[df['total_episode_idx'].isin(train_episodes)].copy()
val_df = df[df['total_episode_idx'].isin(val_episodes)].copy()

X_train = train_df[FEATURES]
y_train = train_df['needs_regeneration'].astype(int)
X_val = val_df[FEATURES]
y_val = val_df['needs_regeneration'].astype(int)

print('Dataset rows:', len(df), 'episodes:', df['total_episode_idx'].nunique())
print('Train rows:', len(train_df), 'episodes:', train_df['total_episode_idx'].nunique(), 'positive rate:', y_train.mean())
print('Val rows:', len(val_df), 'episodes:', val_df['total_episode_idx'].nunique(), 'positive rate:', y_val.mean())
print('Features:', len(FEATURES))

def evaluate_predictions(name: str, y_true: pd.Series, prob: np.ndarray, threshold: float = 0.5) -> dict[str, float | str]:
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        'model': name,
        'threshold': threshold,
        'roc_auc': roc_auc_score(y_true, prob),
        'average_precision': average_precision_score(y_true, prob),
        'accuracy': accuracy_score(y_true, pred),
        'precision_failure': precision_score(y_true, pred, zero_division=0),
        'recall_failure': recall_score(y_true, pred, zero_division=0),
        'f1_failure': f1_score(y_true, pred, zero_division=0),
        'specificity_success': tn / max(tn + fp, 1),
        'regen_rate': float(pred.mean()),
        'brier': brier_score_loss(y_true, prob),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }

def threshold_table(name: str, y_true: pd.Series, prob: np.ndarray) -> pd.DataFrame:
    rows = []
    for threshold in np.round(np.linspace(0.02, 0.98, 97), 3):
        rows.append(evaluate_predictions(name, y_true, prob, float(threshold)))
    table = pd.DataFrame(rows)
    table['objective_recall80_regen35'] = np.where(
        (table['recall_failure'] >= 0.80) & (table['regen_rate'] <= 0.35),
        table['f1_failure'],
        -1.0,
    )
    table['objective_recall90_regen50'] = np.where(
        (table['recall_failure'] >= 0.90) & (table['regen_rate'] <= 0.50),
        table['f1_failure'],
        -1.0,
    )
    return table

logreg = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=5000, class_weight='balanced', solver='lbfgs', random_state=RANDOM_STATE)),
    ]
)
logreg.fit(X_train, y_train)
logreg_prob = logreg.predict_proba(X_val)[:, 1]

catboost = CatBoostClassifier(
    iterations=1200,
    learning_rate=0.03,
    depth=4,
    l2_leaf_reg=6,
    loss_function='Logloss',
    eval_metric='AUC',
    auto_class_weights='Balanced',
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False,
    od_type='Iter',
    od_wait=80,
)
catboost.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
catboost_prob = catboost.predict_proba(X_val)[:, 1]

summary = pd.DataFrame([
    evaluate_predictions('logistic_regression', y_val, logreg_prob, 0.5),
    evaluate_predictions('catboost', y_val, catboost_prob, 0.5),
])
summary.to_csv(OUTPUT_DIR / 'validation_metrics_at_0p5.csv', index=False)
display(summary)

thresholds = pd.concat([
    threshold_table('logistic_regression', y_val, logreg_prob),
    threshold_table('catboost', y_val, catboost_prob),
], ignore_index=True)
thresholds.to_csv(OUTPUT_DIR / 'threshold_sweep.csv', index=False)

recommended_rows = []
for model_name, group in thresholds.groupby('model'):
    for objective_col, label in [
        ('objective_recall80_regen35', 'recall>=0.80 and regen_rate<=0.35'),
        ('objective_recall90_regen50', 'recall>=0.90 and regen_rate<=0.50'),
    ]:
        feasible = group[group[objective_col] >= 0].copy()
        if feasible.empty:
            best = group.sort_values(['f1_failure', 'recall_failure'], ascending=False).iloc[0].copy()
            best['selection_rule'] = f'fallback max f1; no threshold satisfied {label}'
        else:
            best = feasible.sort_values([objective_col, 'precision_failure'], ascending=False).iloc[0].copy()
            best['selection_rule'] = label
        recommended_rows.append(best)
recommended = pd.DataFrame(recommended_rows)
recommended.to_csv(OUTPUT_DIR / 'recommended_thresholds.csv', index=False)
display(recommended[['model', 'selection_rule', 'threshold', 'roc_auc', 'accuracy', 'precision_failure', 'recall_failure', 'f1_failure', 'specificity_success', 'regen_rate', 'tp', 'fp', 'tn', 'fn']])

val_predictions = val_df[['total_episode_idx', 'task_id', 'episode_idx', 'chunk_id', 'query_timestep', 'success', 'needs_regeneration']].copy()
val_predictions['logistic_regression_prob'] = logreg_prob
val_predictions['catboost_prob'] = catboost_prob
val_predictions.to_csv(OUTPUT_DIR / 'validation_chunk_predictions.csv', index=False)

coef = logreg.named_steps['clf'].coef_[0]
logreg_importance = pd.DataFrame({'feature': FEATURES, 'coefficient': coef, 'abs_coefficient': np.abs(coef)})
logreg_importance.sort_values('abs_coefficient', ascending=False).to_csv(OUTPUT_DIR / 'logreg_coefficients.csv', index=False)
catboost_importance = pd.DataFrame({'feature': FEATURES, 'importance': catboost.get_feature_importance()})
catboost_importance.sort_values('importance', ascending=False).to_csv(OUTPUT_DIR / 'catboost_feature_importance.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for model_name, prob in [('logistic_regression', logreg_prob), ('catboost', catboost_prob)]:
    fpr, tpr, _ = roc_curve(y_val, prob)
    axes[0].plot(fpr, tpr, label=f'{model_name} AUC={roc_auc_score(y_val, prob):.3f}')
    precision, recall, _ = precision_recall_curve(y_val, prob)
    axes[1].plot(recall, precision, label=f'{model_name} AP={average_precision_score(y_val, prob):.3f}')
    model_thresholds = thresholds[thresholds['model'] == model_name]
    axes[2].plot(model_thresholds['threshold'], model_thresholds['recall_failure'], label=f'{model_name} recall')
    axes[2].plot(model_thresholds['threshold'], model_thresholds['regen_rate'], linestyle='--', label=f'{model_name} regen rate')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[0].set_title('ROC')
axes[0].set_xlabel('false positive rate on success chunks')
axes[0].set_ylabel('recall on failure chunks')
axes[1].set_title('Precision-Recall')
axes[1].set_xlabel('recall_failure')
axes[1].set_ylabel('precision_failure')
axes[2].set_title('Threshold tradeoff')
axes[2].set_xlabel('threshold')
axes[2].set_ylabel('value')
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'validation_curves.png', dpi=160, bbox_inches='tight')
plt.show()

report = {
    'metrics_dir': str(METRICS_DIR),
    'output_dir': str(OUTPUT_DIR),
    'target': 'needs_regeneration = 1 for chunks from failure episodes',
    'split': 'episode-level stratified train/validation split',
    'n_rows': int(len(df)),
    'n_episodes': int(df['total_episode_idx'].nunique()),
    'n_train_rows': int(len(train_df)),
    'n_val_rows': int(len(val_df)),
    'features': FEATURES,
    'validation_metrics_at_0p5': summary.to_dict(orient='records'),
    'recommended_thresholds': recommended.to_dict(orient='records'),
}
(OUTPUT_DIR / 'training_report.json').write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved outputs to:', OUTPUT_DIR)
